# Experiment 05: Transfer Learning with ResNet18

In the previous experiments, I trained custom CNN models from scratch. The best custom CNN was the VGG-style model from Experiment 04, which achieved 63.70% best validation accuracy.

In this experiment, I use transfer learning with ResNet18. ResNet18 was originally trained on a large image dataset, so it already has useful visual feature extraction layers.

Since the FER images are 48×48 grayscale images, I modify the first convolutional layer of ResNet18 to accept one input channel instead of three. I also replace the final classification layer so that the model predicts 7 emotion classes.

The goal of this experiment is to check whether transfer learning can improve performance compared to my best custom CNN.

In [4]:
!pip install kaggle wandb -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 9.0 MB/s eta 0:00:00


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/ML_Assignment4/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [7]:
!ls -la ~/.kaggle

total 16
drwxr-xr-x 2 root root 4096 Jun 17 12:28 .
drwx------ 1 root root 4096 Jun 17 12:28 ..
-rw------- 1 root root   63 Jun 17 12:28 kaggle.json


In [8]:
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge

100% 285M/285M [00:02<00:00, 118MB/s]



In [9]:
!mkdir -p data
!unzip -q challenges-in-representation-learning-facial-expression-recognition-challenge.zip -d data
!ls data

example_submission.csv	fer2013.tar.gz	icml_face_data.csv  test.csv  train.csv


In [10]:
import pandas as pd

df = pd.read_csv("data/train.csv")

print(df.shape)
print(df.columns)
df.head()

(28709, 2)
Index(['emotion', 'pixels'], dtype='object')


,emotion,pixels
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1,0,151 150 147 155 148 133 111 140 170 174 182 15...
2,2,231 212 156 164 174 138 161 173 182 200 106 38...
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...


In [11]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gvakh23 (gvakh23-tbilisi-free-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [12]:
from torchvision import transforms, models

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device available:", device)

Device available: cpu


In [13]:
import pandas as pd

print(df.shape)
print(df.columns)
df.head()

(28709, 2)
Index(['emotion', 'pixels'], dtype='object')


,emotion,pixels
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1,0,151 150 147 155 148 133 111 140 170 174 182 15...
2,2,231 212 156 164 174 138 161 173 182 200 106 38...
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...


In [14]:
emotion_map = {
    0: "Angry",
    1: "Disgust",
    2: "Fear",
    3: "Happy",
    4: "Sad",
    5: "Surprise",
    6: "Neutral"
}

df["emotion_name"] = df["emotion"].map(emotion_map)
df.head()

,emotion,pixels,emotion_name
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...,Angry
1,0,151 150 147 155 148 133 111 140 170 174 182 15...,Angry
2,2,231 212 156 164 174 138 161 173 182 200 106 38...,Fear
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...,Sad
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...,Neutral


In [15]:
def pixels_to_image(pixels_str):
    pixels = np.array(pixels_str.split(), dtype=np.uint8)
    image = pixels.reshape(48, 48)
    return image

df["pixels_array"] = df["pixels"].apply(pixels_to_image)

print(type(df["pixels_array"].iloc[0]))
print(df["pixels_array"].iloc[0].shape)

<class 'numpy.ndarray'>
(48, 48)


In [16]:
X = np.stack(df["pixels_array"].values)
y = df["emotion"].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test_new, y_val, y_test_new = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("New test size:", len(X_test_new))

Train size: 20096
Validation size: 4306
New test size: 4307


In [17]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [18]:
class EmotionDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = np.array(images)
        self.labels = np.array(labels)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = int(self.labels[idx])

        if self.transform is not None:
            image = self.transform(image)
        else:
            image = torch.tensor(image, dtype=torch.float32).unsqueeze(0) / 255.0

        label = torch.tensor(label, dtype=torch.long)

        return image, label

In [19]:
batch_size = 64

train_dataset = EmotionDataset(
    X_train,
    y_train,
    transform=train_transform
)

val_dataset = EmotionDataset(
    X_val,
    y_val,
    transform=eval_transform
)

test_dataset = EmotionDataset(
    X_test_new,
    y_test_new,
    transform=eval_transform
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Number of train batches:", len(train_dataloader))
print("Number of validation batches:", len(val_dataloader))
print("Number of test batches:", len(test_dataloader))

Number of train batches: 314
Number of validation batches: 68
Number of test batches: 68


In [20]:
images, labels = next(iter(train_dataloader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("Min pixel:", images.min().item())
print("Max pixel:", images.max().item())
print("First 10 labels:", labels[:10])

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Image batch shape: torch.Size([64, 1, 48, 48])
Label batch shape: torch.Size([64])
Image dtype: torch.float32
Label dtype: torch.int64
Min pixel: -1.0
Max pixel: 1.0
First 10 labels: tensor([3, 3, 4, 5, 4, 2, 0, 6, 6, 2])


In [21]:
class ResNet18GrayscaleSmallInput(nn.Module):
    def __init__(self, num_classes=7, dropout_rate=0.4):
        super(ResNet18GrayscaleSmallInput, self).__init__()

        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        old_conv = self.resnet.conv1

        # For 48x48 images, use a smaller 3x3 conv with stride 1.
        # This avoids losing spatial information too early.
        self.resnet.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        # Remove the first maxpool layer.
        # Original ResNet downsamples too aggressively for 48x48 FER images.
        self.resnet.maxpool = nn.Identity()

        # Initialize the new grayscale 3x3 conv using the center of pretrained RGB weights.
        with torch.no_grad():
            grayscale_weights = old_conv.weight.mean(dim=1, keepdim=True)  # [64, 1, 7, 7]
            center_3x3 = grayscale_weights[:, :, 2:5, 2:5]                # [64, 1, 3, 3]
            self.resnet.conv1.weight.copy_(center_3x3)

        num_features = self.resnet.fc.in_features

        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.resnet(x)

In [33]:
WANDB_PROJECT = "Facial_Expression_Recognition"

config = {
    "learning_rate_stage1": 0.001,
    "learning_rate_stage2": 0.00005,

    "stage1_epochs": 5,
    "stage2_epochs": 20,
    "batch_size": 64,

    "optimizer": "AdamW",
    "loss_function": "CrossEntropyLoss_label_smoothing",
    "architecture": "ResNet18GrayscaleSmallInput_TwoStageFineTuning",

    "pretrained": True,
    "transfer_learning": True,
    "input_size": "48x48",
    "input_channels": 1,
    "classifier_hidden_dim": 256,

    "augmentation": True,
    "augmentation_type": "horizontal_flip_rotation_translation_scale",

    "dropout_rate": 0.4,
    "weight_decay": 5e-4,
    "label_smoothing": 0.05,

    "stage1_trainable_layers": "classifier only",
    "stage2_trainable_layers": "layer3 + layer4 + classifier",

    "gradient_clipping": True,
    "max_grad_norm": 1.0
}

run = wandb.init(
    project=WANDB_PROJECT,
    name="exp05_resnet18_twostage_finetune",
    config=config
)

In [34]:
model = ResNet18GrayscaleSmallInput(
    num_classes=7,
    dropout_rate=config["dropout_rate"]
).to(device)

criterion = nn.CrossEntropyLoss(
    label_smoothing=config["label_smoothing"]
)

# Stage 1: freeze the whole ResNet feature extractor
for param in model.resnet.parameters():
    param.requires_grad = False

# Keep only the final classifier trainable
for param in model.resnet.fc.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config["learning_rate_stage1"],
    weight_decay=config["weight_decay"]
)

num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
num_total = sum(p.numel() for p in model.parameters())

wandb.config.update({
    "stage1_trainable_parameters": num_trainable,
    "total_parameters": num_total
})

print(model)
print("Stage 1 trainable parameters:", num_trainable)
print("Total parameters:", num_total)

ResNet18GrayscaleSmallInput(
  (resnet): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): Identity()
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace

In [35]:
def train_epoch(model, dataloader, criterion, optimizer, device, max_grad_norm=None):
    model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()

        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=max_grad_norm
            )

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predicted_labels = outputs.argmax(dim=1)
        correct_predictions += (predicted_labels == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples

    return epoch_loss, epoch_accuracy

In [36]:
def validate_epoch(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)

            predictions = outputs.argmax(dim=1)
            correct_predictions += (predictions == labels).sum().item()
            total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples

    return epoch_loss, epoch_accuracy

In [37]:
# Forward pass sanity check

model.eval()

images, labels = next(iter(train_dataloader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)

print("Input batch shape:", images.shape)
print("Output shape:", outputs.shape)
print("Labels shape:", labels.shape)

assert outputs.shape[0] == images.shape[0], "Batch size mismatch"
assert outputs.shape[1] == 7, "Model output should have 7 logits"
assert labels.ndim == 1, "Labels should be a 1D tensor"
assert torch.isfinite(outputs).all(), "Output contains NaN or Inf"

wandb.log({
    "forward_check_passed": 1,
    "forward_batch_size": images.shape[0],
    "forward_num_classes": outputs.shape[1]
})

print("Forward pass check passed.")

Input batch shape: torch.Size([64, 1, 48, 48])
Output shape: torch.Size([64, 7])
Labels shape: torch.Size([64])
Forward pass check passed.


In [38]:
# Backward pass and gradient sanity check

model.train()

images, labels = next(iter(train_dataloader))
images = images.to(device)
labels = labels.to(device)

optimizer.zero_grad()

outputs = model(images)
loss = criterion(outputs, labels)

print("Initial loss:", loss.item())

loss.backward()

total_grad_norm = 0.0
num_params_with_grad = 0

for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        grad_norm = param.grad.data.norm(2).item()
        total_grad_norm += grad_norm
        num_params_with_grad += 1

print("Number of parameters with gradients:", num_params_with_grad)
print("Total gradient norm:", total_grad_norm)

assert torch.isfinite(loss), "Loss is NaN or Inf"
assert num_params_with_grad > 0, "No gradients were computed"
assert total_grad_norm > 0, "Gradient norm is zero"

wandb.log({
    "backward_check_passed": 1,
    "sanity_initial_loss": loss.item(),
    "sanity_num_params_with_grad": num_params_with_grad,
    "sanity_total_grad_norm": total_grad_norm
})

optimizer.zero_grad()

print("Backward pass check passed.")

Initial loss: 2.047041654586792
Number of parameters with gradients: 6
Total gradient norm: 4.917279121492406
Backward pass check passed.


In [39]:
# One-batch overfitting check

debug_model = ResNet18GrayscaleSmallInput(
    num_classes=7,
    dropout_rate=config["dropout_rate"]
).to(device)

# For the debug model, train only the classifier first, same as Stage 1
for param in debug_model.resnet.parameters():
    param.requires_grad = False

for param in debug_model.resnet.fc.parameters():
    param.requires_grad = True

debug_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, debug_model.parameters()),
    lr=0.001,
    weight_decay=0.0
)

debug_criterion = nn.CrossEntropyLoss()

debug_images, debug_labels = next(iter(train_dataloader))
debug_images = debug_images.to(device)
debug_labels = debug_labels.to(device)

debug_model.train()

for step in range(80):
    debug_optimizer.zero_grad()

    outputs = debug_model(debug_images)
    loss = debug_criterion(outputs, debug_labels)

    loss.backward()
    debug_optimizer.step()

    preds = outputs.argmax(dim=1)
    acc = (preds == debug_labels).float().mean().item()

    if (step + 1) % 20 == 0:
        print(f"Step {step+1}/80 | Loss: {loss.item():.4f} | Accuracy: {acc:.4f}")

wandb.log({
    "one_batch_overfit_final_loss": loss.item(),
    "one_batch_overfit_final_accuracy": acc
})

print("Final one-batch accuracy:", acc)
print("One-batch overfitting check finished.")

Step 20/80 | Loss: 0.5913 | Accuracy: 0.9219
Step 40/80 | Loss: 0.2479 | Accuracy: 0.9688
Step 60/80 | Loss: 0.1448 | Accuracy: 1.0000
Step 80/80 | Loss: 0.0802 | Accuracy: 1.0000
Final one-batch accuracy: 1.0
One-batch overfitting check finished.


## Training Loop


In [40]:
stage1_epochs = config["stage1_epochs"]

for epoch in range(stage1_epochs):
    train_loss, train_accuracy = train_epoch(
        model,
        train_dataloader,
        criterion,
        optimizer,
        device,
        max_grad_norm=config["max_grad_norm"]
    )

    val_loss, val_accuracy = validate_epoch(
        model,
        val_dataloader,
        criterion,
        device
    )

    current_lr = optimizer.param_groups[0]["lr"]

    wandb.log({
        "stage": 1,
        "stage_epoch": epoch + 1,
        "global_epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "learning_rate": current_lr
    })

    print(
        f"Stage 1 Epoch {epoch+1}/{stage1_epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f} | "
        f"LR: {current_lr:.6f}"
    )

Stage 1 Epoch 1/5 | Train Loss: 1.7529, Train Acc: 0.3082 | Val Loss: 1.6470, Val Acc: 0.3841 | LR: 0.001000
Stage 1 Epoch 2/5 | Train Loss: 1.7083, Train Acc: 0.3274 | Val Loss: 1.6398, Val Acc: 0.3781 | LR: 0.001000
Stage 1 Epoch 3/5 | Train Loss: 1.6844, Train Acc: 0.3438 | Val Loss: 1.6279, Val Acc: 0.3892 | LR: 0.001000
Stage 1 Epoch 4/5 | Train Loss: 1.6816, Train Acc: 0.3460 | Val Loss: 1.6217, Val Acc: 0.3841 | LR: 0.001000
Stage 1 Epoch 5/5 | Train Loss: 1.6761, Train Acc: 0.3470 | Val Loss: 1.6200, Val Acc: 0.3790 | LR: 0.001000


In [41]:
# Stage 2: unfreeze only later ResNet blocks and classifier

for param in model.resnet.layer3.parameters():
    param.requires_grad = True

for param in model.resnet.layer4.parameters():
    param.requires_grad = True

for param in model.resnet.fc.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config["learning_rate_stage2"],
    weight_decay=config["weight_decay"]
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

wandb.config.update({
    "stage2_trainable_parameters": num_trainable
})

print("Stage 2 trainable parameters:", num_trainable)

Stage 2 trainable parameters: 10627079


In [42]:
!mkdir -p /content/drive/MyDrive/ML_Assignment4/models

best_model_path = "/content/drive/MyDrive/ML_Assignment4/models/exp05_resnet18_twostage_finetune.pt"

stage2_epochs = config["stage2_epochs"]
best_val_accuracy = 0.0
best_val_loss = float("inf")

global_epoch_offset = config["stage1_epochs"]

for epoch in range(stage2_epochs):
    train_loss, train_accuracy = train_epoch(
        model,
        train_dataloader,
        criterion,
        optimizer,
        device,
        max_grad_norm=config["max_grad_norm"]
    )

    val_loss, val_accuracy = validate_epoch(
        model,
        val_dataloader,
        criterion,
        device
    )

    scheduler.step(val_accuracy)

    current_lr = optimizer.param_groups[0]["lr"]
    global_epoch = global_epoch_offset + epoch + 1

    wandb.log({
        "stage": 2,
        "stage_epoch": epoch + 1,
        "global_epoch": global_epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "learning_rate": current_lr
    })

    print(
        f"Stage 2 Epoch {epoch+1}/{stage2_epochs} | "
        f"Global Epoch {global_epoch} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print("Saved best model to:", best_model_path)

print("Best validation accuracy:", best_val_accuracy)
print("Best validation loss:", best_val_loss)

wandb.log({
    "best_val_accuracy": best_val_accuracy,
    "best_val_loss": best_val_loss
})

wandb.save(best_model_path)
wandb.finish()
print("WandB run finished.")

Stage 2 Epoch 1/20 | Global Epoch 6 | Train Loss: 1.4600, Train Acc: 0.4682 | Val Loss: 1.3103, Val Acc: 0.5307 | LR: 0.000050
Saved best model to: /content/drive/MyDrive/ML_Assignment4/models/exp05_resnet18_twostage_finetune.pt
Stage 2 Epoch 2/20 | Global Epoch 7 | Train Loss: 1.2703, Train Acc: 0.5608 | Val Loss: 1.2250, Val Acc: 0.5685 | LR: 0.000050
Saved best model to: /content/drive/MyDrive/ML_Assignment4/models/exp05_resnet18_twostage_finetune.pt
Stage 2 Epoch 3/20 | Global Epoch 8 | Train Loss: 1.1668, Train Acc: 0.6108 | Val Loss: 1.1861, Val Acc: 0.5889 | LR: 0.000050
Saved best model to: /content/drive/MyDrive/ML_Assignment4/models/exp05_resnet18_twostage_finetune.pt
Stage 2 Epoch 4/20 | Global Epoch 9 | Train Loss: 1.0871, Train Acc: 0.6480 | Val Loss: 1.1701, Val Acc: 0.5959 | LR: 0.000050
Saved best model to: /content/drive/MyDrive/ML_Assignment4/models/exp05_resnet18_twostage_finetune.pt
Stage 2 Epoch 5/20 | Global Epoch 10 | Train Loss: 1.0069, Train Acc: 0.6836 | Val L

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Stage 2 Epoch 20/20 | Global Epoch 25 | Train Loss: 0.3837, Train Acc: 0.9664 | Val Loss: 1.3477, Val Acc: 0.6256 | LR: 0.000025
Best validation accuracy: 0.6293543892243382
Best validation loss: 1.3267680395773496


backward_check_passed,▁
best_val_accuracy,▁
best_val_loss,▁
forward_batch_size,▁
forward_check_passed,▁
forward_num_classes,▁
global_epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
learning_rate,█████▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
one_batch_overfit_final_accuracy,▁
one_batch_overfit_final_loss,▁
+9,...


WandB run finished.


In [22]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import numpy as np
import torch
import wandb

def evaluate_model_detailed(model, dataloader, criterion, device, emotion_map):
    model.eval()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)

            predictions = outputs.argmax(dim=1)
            correct_predictions += (predictions == labels).sum().item()
            total_samples += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    test_loss = running_loss / total_samples
    test_accuracy = correct_predictions / total_samples

    cm = confusion_matrix(all_labels, all_predictions)
    report = classification_report(
        all_labels,
        all_predictions,
        target_names=[emotion_map[i] for i in range(7)],
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).transpose()

    print("Test loss:", test_loss)
    print("Test accuracy:", test_accuracy)
    print("\nClassification report:")
    print(report_df)

    return test_loss, test_accuracy, cm, report_df, all_labels, all_predictions


In [23]:
# RUN FOR FINAL TEST: YES.
# Final test evaluation for Experiment 05 with WandB logging

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "/content/drive/MyDrive/ML_Assignment4/models/exp05_resnet18_twostage_finetune.pt"

# Recreate the exact Experiment 05 architecture before loading weights
test_model = ResNet18GrayscaleSmallInput(
    num_classes=7,
    dropout_rate=0.4
).to(device)

test_model.load_state_dict(
    torch.load(checkpoint_path, map_location=device)
)

test_model.eval()

test_criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

# Close any active run before starting a clean final test run
if wandb.run is not None:
    wandb.finish()

test_run = wandb.init(
    project="Facial_Expression_Recognition",
    name="exp05_resnet18_twostage_test_evaluation",
    config={
        "architecture": "ResNet18GrayscaleSmallInput_TwoStageFineTuning",
        "experiment": "Experiment 05",
        "pretrained": True,
        "transfer_learning": True,
        "fine_tuning_strategy": "two_stage",
        "stage1": "classifier only",
        "stage2": "layer3 + layer4 + classifier",
        "input_size": "48x48",
        "input_channels": 1,
        "dropout_rate": 0.4,
        "label_smoothing": 0.05,
        "checkpoint_path": checkpoint_path,
        "evaluation_split": "internal_test",
        "loaded_from_checkpoint": True
    }
)

# Forward and backward sanity checks on the loaded checkpoint
sanity_inputs, sanity_labels = next(iter(train_dataloader))
sanity_inputs = sanity_inputs.to(device)
sanity_labels = sanity_labels.to(device)

test_model.eval()
with torch.no_grad():
    sanity_outputs = test_model(sanity_inputs)

print("Sanity input shape:", sanity_inputs.shape)
print("Sanity output shape:", sanity_outputs.shape)
print("Sanity labels shape:", sanity_labels.shape)

assert sanity_outputs.shape[0] == sanity_inputs.shape[0], "Batch size mismatch"
assert sanity_outputs.shape[1] == 7, "Model output should have 7 logits"
assert sanity_labels.ndim == 1, "Labels should be 1D"
assert torch.isfinite(sanity_outputs).all(), "Outputs contain NaN or Inf"

print("Forward pass check passed.")

test_model.train()
test_model.zero_grad()

sanity_outputs = test_model(sanity_inputs)
sanity_loss = test_criterion(sanity_outputs, sanity_labels)

print("Sanity loss:", sanity_loss.item())

sanity_loss.backward()

total_grad_norm = 0.0
num_params_with_grad = 0

for name, param in test_model.named_parameters():
    if param.requires_grad and param.grad is not None:
        grad_norm = param.grad.data.norm(2).item()
        total_grad_norm += grad_norm
        num_params_with_grad += 1

print("Number of parameters with gradients:", num_params_with_grad)
print("Total gradient norm:", total_grad_norm)

assert torch.isfinite(sanity_loss), "Loss is NaN or Inf"
assert num_params_with_grad > 0, "No gradients were computed"
assert total_grad_norm > 0, "Gradient norm is zero"

test_model.zero_grad()
test_model.eval()

wandb.log({
    "forward_check_passed": 1,
    "backward_check_passed": 1,
    "sanity_initial_loss": sanity_loss.item(),
    "sanity_num_params_with_grad": num_params_with_grad,
    "sanity_total_grad_norm": total_grad_norm
})

print("Backward pass check passed.")

# Final internal test evaluation
test_loss, test_accuracy, cm, report_df, all_labels, all_predictions = evaluate_model_detailed(
    test_model,
    test_dataloader,
    test_criterion,
    device,
    emotion_map
)

wandb.log({
    "test_loss": test_loss,
    "test_accuracy": test_accuracy
})

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        y_true=all_labels,
        preds=all_predictions,
        class_names=[emotion_map[i] for i in range(7)]
    )
})

wandb.log({
    "classification_report": wandb.Table(dataframe=report_df.reset_index())
})

wandb.finish()
print("Experiment 05 test evaluation finished.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 131MB/s]


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Sanity input shape: torch.Size([64, 1, 48, 48])
Sanity output shape: torch.Size([64, 7])
Sanity labels shape: torch.Size([64])
Forward pass check passed.
Sanity loss: 0.38381844758987427
Number of parameters with gradients: 66
Total gradient norm: 46.817644567566916
Backward pass check passed.
Test loss: 1.312956818660783
Test accuracy: 0.635244950081263

Classification report:
              precision    recall  f1-score      support
Angry          0.533220  0.522538  0.527825   599.000000
Disgust        0.606061  0.615385  0.610687    65.000000
Fear           0.518868  0.447154  0.480349   615.000000
Happy          0.821269  0.848569  0.834696  1083.000000
Sad            0.498705  0.531034  0.514362   725.000000
Surprise       0.747475  0.778947  0.762887   475.000000
Neutral        0.588076  0.582550  0.585300   745.000000
accuracy       0.635245  0.635245  0.635245     0.635245
macro avg      0.616239  0.618025  0.616586  4307.000000
weighted avg   0.632008  0.635245  0.633059  4307

backward_check_passed,▁
forward_check_passed,▁
sanity_initial_loss,▁
sanity_num_params_with_grad,▁
sanity_total_grad_norm,▁
test_accuracy,▁
test_loss,▁
backward_check_passed,1
forward_check_passed,1
sanity_initial_loss,0.38382
sanity_num_params_with_grad,66


Experiment 05 test evaluation finished.
